# This Notebook implements 2 different methods of hyperparameter tuning
# Namely **(Grid Search CV)** and **(Randomized Search CV)**

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = sns.load_dataset('iris')

In [3]:
df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [4]:
df['species'].unique()

array(['setosa', 'versicolor', 'virginica'], dtype=object)

In [5]:
from sklearn.model_selection import train_test_split


In [6]:
X = df.drop(columns='species')
y = df['species']

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [8]:
from sklearn.neighbors import KNeighborsClassifier as KNN

In [9]:
knn = KNN(n_neighbors=13)

In [10]:
knn.fit(X_train,y_train)

,n_neighbors,13
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [11]:
knn.score(X_test,y_test)

1.0

In [12]:
from sklearn.svm import SVC

In [13]:
# svm = SVC(gamma='auto',kernel='rbf',C=30)
svm = SVC(gamma='auto',kernel='linear',C=10)

In [14]:
svm.fit(X_train,y_train)

,C,10
,kernel,'linear'
,degree,3
,gamma,'auto'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [15]:
svm.score(X_test,y_test)

1.0

# Now we will use grid search CV

In [16]:
from sklearn.model_selection import GridSearchCV
classifier = GridSearchCV(svm,{
    'C': [1,10,20,30],
    'kernel':['linear','rbf']
},cv = 5,return_train_score = False)

In [17]:
classifier.fit(X_train,y_train)

,estimator,"SVC(C=10, gam...rnel='linear')"
,param_grid,"{'C': [1, 10, ...], 'kernel': ['linear', 'rbf']}"
,scoring,None
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,C,1


In [21]:
results = pd.DataFrame(classifier.cv_results_)
classifier.cv_results_

{'mean_fit_time': array([0.00386562, 0.00321198, 0.00272064, 0.00271006, 0.00321131,
        0.00190129, 0.00121388, 0.00232162]),
 'std_fit_time': array([0.00154689, 0.00039388, 0.00039715, 0.00040197, 0.000248  ,
        0.00109924, 0.00039977, 0.00097778]),
 'mean_score_time': array([0.00302119, 0.0030643 , 0.00253339, 0.00228782, 0.00191436,
        0.00119619, 0.00079451, 0.00161891]),
 'std_score_time': array([0.00115578, 0.00026351, 0.00058345, 0.00039246, 0.00017776,
        0.00040036, 0.00039745, 0.00050294]),
 'param_C': masked_array(data=[1, 1, 10, 10, 20, 20, 30, 30],
              mask=[False, False, False, False, False, False, False, False],
        fill_value=999999),
 'param_kernel': masked_array(data=['linear', 'rbf', 'linear', 'rbf', 'linear', 'rbf',
                    'linear', 'rbf'],
              mask=[False, False, False, False, False, False, False, False],
        fill_value='?',
             dtype=object),
 'params': [{'C': 1, 'kernel': 'linear'},
  {'C': 1, 

In [67]:
results[['param_C','param_kernel','mean_test_score']]

,param_C,param_kernel,mean_test_score
0,1,linear,0.95
1,1,rbf,0.94
2,10,linear,0.93
3,10,rbf,0.93
4,20,linear,0.93
5,20,rbf,0.92
6,30,linear,0.94
7,30,rbf,0.93


# Random Search CV

In [28]:
from sklearn.model_selection import RandomizedSearchCV
classifier2 = RandomizedSearchCV(svm,{
    'C':[1,10,20,30],
    'kernel':['linear','rbf']
},n_iter=4,cv=5,return_train_score=False)

In [29]:
classifier2.fit(X_train,y_train)

,estimator,"SVC(C=10, gam...rnel='linear')"
,param_distributions,"{'C': [1, 10, ...], 'kernel': ['linear', 'rbf']}"
,n_iter,4
,scoring,None
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,None
,error_score,nan


In [30]:
results = pd.DataFrame(classifier2.cv_results_)
results[['mean_test_score','param_C','param_kernel']]

,mean_test_score,param_C,param_kernel
0,0.93,30,rbf
1,0.92,20,rbf
2,0.94,30,linear
3,0.93,10,rbf


In [31]:
results

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_kernel,param_C,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.004462,0.000413,0.003048,0.000637,rbf,30,"{'kernel': 'rbf', 'C': 30}",0.95,0.80,0.9,1.0,1.00,0.93,0.074833,2
1,0.003691,0.000574,0.002756,0.000203,rbf,20,"{'kernel': 'rbf', 'C': 20}",0.95,0.80,0.9,1.0,0.95,0.92,0.067823,4
2,0.003367,0.000450,0.001999,0.000001,linear,30,"{'kernel': 'linear', 'C': 30}",0.95,0.85,0.9,1.0,1.00,0.94,0.058310,1
3,0.002693,0.000433,0.002204,0.000390,rbf,10,"{'kernel': 'rbf', 'C': 10}",0.95,0.80,0.9,1.0,1.00,0.93,0.074833,2
